In [ ]:
import numpy as np
import pandas as pd
import os
import glob

def process_weather_data():
    # 1. 查找所有 XLS Excel 文件
    files = glob.glob('*.xls')

    # 用来存放所有处理好的数据片段
    all_data_frames = []
    processed_count = 0

    print(f"找到 {len(files)} 个 xls 文件，准备开始处理...")

    for file_path in files:
        # 跳过已经处理过的文件 (以 processed_ 或 merged_ 开头)
        if file_path.startswith('processed_') or file_path.startswith('merged_'):
            continue
        # 跳过香港的对照数据文件
        if '香港' in file_path:
            continue

        try:
            # 读取数据
            df = pd.read_excel(file_path)

            # -------------------------------------------------------
            # 2. 智能识别：检查是否是目标气象数据文件
            # -------------------------------------------------------
            required_columns = ['参考地名', '海平面气压(hPa)', '平均气温2m(℃)']
            if not all(col in df.columns for col in required_columns):
                continue

            print(f"正在读取并清洗: {file_path} ...")

            # Initialize the column to ensure it exists for mapping later
            df['盛行風向'] = np.nan

            # -------------------------------------------------------
            # 3. 数据换算 (Unit Conversion)
            # -------------------------------------------------------
            if '风速10m(m/s)' in df.columns:
                df['平均風速 (公里/小時)'] = df['风速10m(m/s)'] * 3.6

            if '经向风速(V,m/s)' in df.columns and '纬向风速(U,m/s)' in df.columns:
                v_comp = df['经向风速(V,m/s)']
                u_comp = df['纬向风速(U,m/s)']

                raw_angle = (270 - np.degrees(np.arctan2(v_comp, u_comp))) % 360
                rounded_angle = (raw_angle / 10).round() * 10

                df['盛行風向'] = (rounded_angle % 360).astype('Int64')

            if '总太阳辐射度(down,J/m2)' in df.columns:
                df['太陽總輻射(兆焦耳平方米)'] = df['总太阳辐射度(down,J/m2)'] / 1000000

            if '总云量(tcc)' in df.columns:
                df['日平均雲量(百分比)'] = df['总云量(tcc)'] * 100

            # -------------------------------------------------------
            # 4. 定义保留字段与重命名映射 (Mapping)
            # -------------------------------------------------------
            columns_map = {
                '日期(UTC)': '日期',
                '参考地名': '地名',
                '经度(lon)': '經度',
                '纬度(lat)': '緯度',
                '平均气温2m(℃)': '日平均氣溫(攝氏度)',
                '最高气温2m(℃)': '日最高氣溫(攝氏度)',
                '最低气温2m(℃)': '日最低氣溫(攝氏度)',
                '地表温度(℃)': '地表温度(℃)',
                '露点温度(℃)': '日平均露點溫度(攝氏度)',
                '相对湿度(%)': '日平均相對濕度(百分比)',
                '降水量(mm)': '日總雨量(毫米)',
                '海平面气压(hPa)': '日平均氣壓(百帕斯卡)',
                '日照时数(峰值,h)': '總日照(小時)',
                '蒸发量(mm)': '日總蒸發量(毫米)',
                '平均風速 (公里/小時)': '平均風速 (公里/小時)',
                '盛行風向': '盛行風向 (0=North)',
                '太陽總輻射(兆焦耳平方米)': '太陽總輻射(兆焦耳平方米)',
                '日平均雲量(百分比)': '日平均雲量(百分比)',
            }

            # -------------------------------------------------------
            # 5. 筛选、重命名并添加到总列表
            # -------------------------------------------------------
            # 找出当前文件中实际存在的需要保留的列
            available_cols = [col for col in columns_map.keys() if col in df.columns]

            df_cleaned = df[available_cols].copy()
            df_cleaned = df_cleaned.rename(columns=columns_map)

            # 格式化小数位
            numeric_cols = df_cleaned.select_dtypes(include=['float64', 'float32']).columns
            df_cleaned[numeric_cols] = df_cleaned[numeric_cols].round(2)

            # 将处理好的这一小份数据加入到列表里
            all_data_frames.append(df_cleaned)
            processed_count += 1

        except Exception as e:
            print(f"跳过文件 {file_path}: 读取错误 ({e})")

    # -------------------------------------------------------
    # 6. 合并所有数据并排序
    # -------------------------------------------------------
    if all_data_frames:
        print("-" * 30)
        print("正在合并所有文件...")

        # 1. 拼接 (Concat)
        merged_df = pd.concat(all_data_frames, ignore_index=True)

        # 2. 排序 (Sort) - 按 '日期' 升序排列
        if '日期' in merged_df.columns:
            merged_df = merged_df.sort_values(by='日期', ascending=True)

        # 3. 导出 (Export)
        location = "阳江" #Change this variable according to the dataset being processed
        output_filename = location+"15-24年天气数据_对齐.csv"
        merged_df.to_csv(output_filename, index=False, encoding='utf-8-sig')

        print(f"成功！共处理 {processed_count} 个文件。")
        print(f"合并后的总数据行数: {len(merged_df)}")
        print(f"时间范围: {merged_df['日期'].min()} 到 {merged_df['日期'].max()}")
        print(f"已保存为: {output_filename}")
    else:
        print("\n未处理任何文件，无法合并。请检查目录下是否有 .xls 文件。")

if __name__ == "__main__":
    process_weather_data()

找到 10 个 xls 文件，准备开始处理...
正在读取并清洗: 22.xls ...
正在读取并清洗: 24.xls ...
正在读取并清洗: 20.xls ...
正在读取并清洗: 23.xls ...
正在读取并清洗: 21.xls ...
正在读取并清洗: 15.xls ...
正在读取并清洗: 18.xls ...
正在读取并清洗: 17.xls ...
正在读取并清洗: 19.xls ...
正在读取并清洗: 16.xls ...
------------------------------
正在合并所有文件...
成功！共处理 10 个文件。
合并后的总数据行数: 3653
时间范围: 20150101 到 20241231
已保存为: 阳江15-24年天气数据_对齐.csv
